# nmtc-mapper — NMTC Eligibility Checker
## Demo: Automated Eligibility Screening with Async Batch Geocoding

This notebook demonstrates how to use nmtc-mapper to:
- Check NMTC eligibility for individual addresses and census tracts
- Batch process thousands of addresses using async geocoding
- Analyze distress level distributions across a project portfolio
- Export results for IC memos and grant applications

Data source: CDFI Fund 2016-2020 ACS Low-Income Community Eligibility File
Geocoding: US Census Bureau Geocoding API (free, no API key required)


In [1]:
# Exercise the INSTALLED nmtc-mapper (no sys.path hack pointing at the source
# tree). Install with `pip install nmtc-mapper` (>= 0.4.0 for tri-state +
# eligibility_status). If this import fails or lacks eligibility_status, the
# installed package is stale — reinstall before trusting the demo.
from nmtcmapper import NMTCMapper, load_sample_table
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

import nmtcmapper
print(f"nmtc-mapper {nmtcmapper.__version__} loaded successfully")


nmtc-mapper 0.5.0 loaded successfully


## 1. Initialize the Mapper

The mapper loads the CDFI Fund eligibility table on startup.
After the first run, it caches the file locally so subsequent
loads are instant.


In [2]:
mapper = NMTCMapper()

print(f"Total census tracts loaded: {mapper.tract_count:,}")
print(f"NMTC eligible tracts:       {mapper.eligible_tract_count:,}")
print(f"Eligibility rate:           {mapper.eligible_tract_count/mapper.tract_count*100:.1f}%")


Loading NMTC eligibility table...
Using cached eligibility file: /Users/jaypatel/.nmtcmapper/cache/NMTC_LIC_Eligibility_2016_2020.xlsb
Loading eligibility table from /Users/jaypatel/.nmtcmapper/cache/NMTC_LIC_Eligibility_2016_2020.xlsb...


Eligibility table loaded: 85,395 census tracts
Ready. 85,395 census tracts loaded.


Loaded 8,764 Opportunity Zone tracts
Opportunity Zones loaded: 8,764 tracts
Total census tracts loaded: 85,395
NMTC eligible tracts:       35,335
Eligibility rate:           41.4%


## 2. Check Individual Census Tracts

The fastest lookup — if you already know the 11-digit FIPS code,
no geocoding is needed.


In [3]:
# All tract IDs below are verified present-and-correct against the real
# 85,395-tract 2016-2020 CDFI Fund universe loaded above — EXCEPT the last one,
# which is deliberately invalid to demonstrate the tri-state UNKNOWN outcome.
# The set is chosen so all three states appear on purpose, not by accident:
#   verified-eligible   -> 17031840100, 17031010100, 01001020200
#   verified-ineligible -> 17031839100, 17031030604
#   indeterminate       -> 99999999999 (absent -> None, never a falsy "NO")
tracts_to_check = [
    ("17031840100", "Chicago, IL — Cook tract 8401.00"),
    ("17031010100", "Chicago, IL — Cook tract 0101.00"),
    ("17031839100", "Chicago, IL — Cook tract 8391.00"),
    ("17031030604", "Chicago, IL — Cook tract 0306.04"),
    ("01001020200", "Autauga County, AL — tract 0202.00"),
    ("99999999999", "deliberately invalid — demonstrates UNKNOWN"),
]

print(f"{'Tract ID':<13} {'Location':<45} {'Eligible':<9} {'Distress':<12} {'Poverty':<9} {'AMI Ratio'}")
print("-" * 100)

for tract_id, location in tracts_to_check:
    result = mapper.check_tract(tract_id)
    # nmtc_eligible is tri-state (True / False / None). Truthiness is WRONG here:
    # `if result.nmtc_eligible` renders an indeterminate None as a fabricated "NO",
    # exactly the bug the 0.4.0 tri-state exists to kill. Compare explicitly.
    if result.nmtc_eligible is True:
        eligible = "YES"
    elif result.nmtc_eligible is False:
        eligible = "NO"
    else:
        eligible = "UNKNOWN"
    # poverty_rate / ami_ratio are Optional[float]; a real 0.0 is falsy, so a
    # truthiness gate would render a genuine 0.0% as "N/A". Gate on `is None`.
    poverty = f"{result.poverty_rate*100:.1f}%" if result.poverty_rate is not None else "N/A"
    ami = f"{result.ami_ratio*100:.1f}%" if result.ami_ratio is not None else "N/A"
    print(f"{tract_id:<13} {location:<45} {eligible:<9} {result.distress_level:<12} {poverty:<9} {ami}")


Tract ID      Location                                      Eligible  Distress     Poverty   AMI Ratio
----------------------------------------------------------------------------------------------------
17031840100   Chicago, IL — Cook tract 8401.00              YES       severe       11.6%     62.5%
17031010100   Chicago, IL — Cook tract 0101.00              YES       lic          27.6%     85.4%
17031839100   Chicago, IL — Cook tract 8391.00              NO        ineligible   10.5%     166.5%
17031030604   Chicago, IL — Cook tract 0306.04              NO        ineligible   19.7%     91.3%
01001020200   Autauga County, AL — tract 0202.00            YES       lic          17.0%     73.6%
99999999999   deliberately invalid — demonstrates UNKNOWN   UNKNOWN   unknown      N/A       N/A


## 3. Distress Level Analysis

Understanding distress levels is critical for NMTC allocation applications.
CDEs must commit at least 85% of resources to Severe Distress or
non-metro LIC tracts.


In [4]:
# PROVENANCE: this distribution is from the SANCTIONED SYNTHETIC sample, NOT the
# real 85,395-tract universe loaded above. We use the public from_sample() /
# load_sample_table() entry points (never the private _build_sample_table), and
# the [SAMPLE] tag rides INLINE on every printed line — a reader can't mistake a
# demo number for real data even if this cell is copied out of context.
sample_mapper = NMTCMapper.from_sample()          # stamps data_source == "sample"
assert sample_mapper.data_source == "sample"
sample_tbl = load_sample_table()                  # public accessor: 12 demo tracts

print(f"Distress distribution — SYNTHETIC SAMPLE "
      f"(data_source={sample_mapper.data_source!r}, {len(sample_tbl)} demo tracts; "
      f"NOT the real universe):")
for level, n in sample_tbl["distress_level"].value_counts().items():
    print(f"  [SAMPLE] {level:<12} {n:>3} tracts")

print(f"\nBy distress level  [SYNTHETIC SAMPLE — demo values, NOT real]:")
for level in ["deep", "severe", "lic", "ineligible"]:
    subset = sample_tbl[sample_tbl["distress_level"] == level]
    if len(subset) > 0:
        avg_poverty = subset["poverty_rate"].mean()
        avg_ami = subset["ami_ratio"].mean()
        print(f"  [SAMPLE] {level:<12} {len(subset):>3} tracts | "
              f"avg poverty: {avg_poverty*100:.1f}% | "
              f"avg AMI: {avg_ami*100:.1f}%")


Distress distribution — SYNTHETIC SAMPLE (data_source='sample', 12 demo tracts; NOT the real universe):
  [SAMPLE] severe         5 tracts
  [SAMPLE] ineligible     4 tracts
  [SAMPLE] deep           2 tracts
  [SAMPLE] lic            1 tracts

By distress level  [SYNTHETIC SAMPLE — demo values, NOT real]:
  [SAMPLE] deep           2 tracts | avg poverty: 43.5% | avg AMI: 46.5%
  [SAMPLE] severe         5 tracts | avg poverty: 32.4% | avg AMI: 63.4%
  [SAMPLE] lic            1 tracts | avg poverty: 22.0% | avg AMI: 78.0%
  [SAMPLE] ineligible     4 tracts | avg poverty: 16.2% | avg AMI: 90.8%


## 4. Batch Processing with Tract IDs

When you have a portfolio of projects with known census tract IDs,
enrich them all at once — no geocoding needed, instant results.


In [5]:
# Same verified tract IDs as the single-lookup table above, so the portfolio
# resolves deliberately (3 eligible / 2 ineligible / 1 indeterminate) rather than
# coming back mostly UNKNOWN. The last row carries a deliberately-invalid tract id
# to show how a stale/bad GEOID in your own data surfaces as indeterminate — NOT
# as a fabricated "ineligible".
portfolio = pd.DataFrame({
    "project_name": [
        "Southside Community Health Center",
        "Near North Mixed-Use",
        "West Side Grocery",
        "Lincoln Park Office",
        "Autauga County Manufacturing",
        "Legacy record — stale tract id",
    ],
    "project_type": [
        "Healthcare", "Mixed-Use", "Food Access",
        "Commercial RE", "Manufacturing", "Data Quality",
    ],
    "amount": [
        3_500_000, 6_000_000, 1_200_000,
        8_000_000, 4_500_000, 750_000,
    ],
    "tract_id": [
        "17031840100", "17031010100", "17031839100",
        "17031030604", "01001020200", "99999999999",
    ],
})

print(f"Portfolio: {len(portfolio)} projects, ${portfolio['amount'].sum()/1e6:.1f}MM total")
print()

enriched = mapper.enrich(portfolio, tract_col="tract_id")
# Include eligibility_status — the additive column 0.4.0 adds to name the four
# outcomes explicitly (verified-eligible / verified-ineligible / not-found /
# geocode-failed) rather than leaving the reader to infer them from None.
print(enriched[["project_name", "nmtc_eligible", "eligibility_status",
                "distress_level", "poverty_rate", "ami_ratio"]].to_string(index=False))


Portfolio: 6 projects, $23.9MM total

Using existing tract IDs from column 'tract_id'
                     project_name nmtc_eligible  eligibility_status distress_level poverty_rate ami_ratio
Southside Community Health Center          True   verified-eligible         severe        0.116   0.62507
             Near North Mixed-Use          True   verified-eligible            lic        0.276  0.853606
                West Side Grocery         False verified-ineligible     ineligible        0.105  1.665149
              Lincoln Park Office         False verified-ineligible     ineligible        0.197  0.912753
     Autauga County Manufacturing          True   verified-eligible            lic         0.17  0.736005
   Legacy record — stale tract id          None           not-found        unknown         None      None


## 5. Portfolio Eligibility Summary

In [6]:
summary = mapper.eligible_count(enriched)

# nmtc_eligible is tri-state (True / False / None). Compare explicitly — a
# `~col` / truthiness gate would fold indeterminate (None) rows into "ineligible".
eligible_portfolio     = enriched[enriched["nmtc_eligible"] == True]
ineligible_portfolio   = enriched[enriched["nmtc_eligible"] == False]
indeterminate_portfolio = enriched[enriched["nmtc_eligible"].isna()]
print(f"Eligible project amount:      ${eligible_portfolio['amount'].sum()/1e6:.1f}MM")
print(f"Ineligible amount:            ${ineligible_portfolio['amount'].sum()/1e6:.1f}MM")
print(f"Indeterminate amount:         ${indeterminate_portfolio['amount'].sum()/1e6:.1f}MM (no match / tract absent — NOT ineligible)")
print(f"\nDistress level breakdown:")
print(enriched.groupby("distress_level")["amount"].agg(["count", "sum"]).to_string())


NMTC Eligibility Summary
  Total addresses:    6
  ── Determined:      5 (verified eligible or verified ineligible)
  ── Indeterminate:   1 (no match / tract absent — NOT ineligible)
  NMTC Eligible:      3 of 5 determined (60.0%)
  ── Deep Distress:   0
  ── Severe Distress: 1
  ── LIC Only:        2
  Not Eligible:       2

Eligible project amount:      $14.0MM
Ineligible amount:            $9.2MM
Indeterminate amount:         $0.8MM (no match / tract absent — NOT ineligible)

Distress level breakdown:
                count       sum
distress_level                 
ineligible          2   9200000
lic                 2  10500000
severe              1   3500000
unknown             1    750000


## 6. Async Batch Geocoding

For large address lists, nmtc-mapper uses async geocoding with:
- `asyncio` + `aiohttp` for concurrent requests
- Semaphore-based rate limiting (max 10 concurrent)
- Exponential backoff retry logic
- Progress bar via `tqdm`

**Note:** Real geocoding requires internet access and may take 1-2 seconds
per address. This demo shows the API — replace with real addresses to run.


In [7]:
from nmtcmapper.geocoder.census import (
    geocode_address, _parse_street, _parse_city, _parse_state, _parse_zip
)

# Show address parsing
sample_address = "1234 S Michigan Ave, Chicago, IL 60605"
print(f"Full address:  {sample_address}")
print(f"Street:        {_parse_street(sample_address)}")
print(f"City:          {_parse_city(sample_address)}")
print(f"State:         {_parse_state(sample_address)}")
print(f"ZIP:           {_parse_zip(sample_address)}")
print()
print("To geocode a real address:")
print("  tract_id = geocode_address('1234 S Michigan Ave, Chicago, IL 60605')")
print()
print("To batch geocode a DataFrame:")
print("  df = mapper.enrich(df, address_col='address')")
print("  # Uses async processing: 10 concurrent requests, retry on failure")


Full address:  1234 S Michigan Ave, Chicago, IL 60605
Street:        1234 S Michigan Ave
City:          Chicago
State:         IL
ZIP:           60605

To geocode a real address:
  tract_id = geocode_address('1234 S Michigan Ave, Chicago, IL 60605')

To batch geocode a DataFrame:
  df = mapper.enrich(df, address_col='address')
  # Uses async processing: 10 concurrent requests, retry on failure


## 7. Eligibility by Project Type

In [8]:
type_summary = enriched.groupby("project_type").agg(
    count=("amount", "count"),
    total_amount=("amount", "sum"),
    # tri-state: count verified-eligible only. A plain "sum" over True/False/None
    # would raise (or silently miscount) now that nmtc_eligible is Optional[bool].
    nmtc_eligible=("nmtc_eligible", lambda s: (s == True).sum()),
    # THE DENOMINATOR, NAMED — the same correction 0.5.0 made to eligible_count().
    # `count` includes indeterminate rows, so `nmtc_eligible / count` reports a
    # verdict for rows nothing was read for. The "Data Quality" row below is the
    # deliberately-invalid tract id: one not-found project, and dividing by
    # `count` published it as 0.0% NMTC eligible.
    determined=("eligibility_status",
                lambda s: s.isin(["verified-eligible", "verified-ineligible"]).sum()),
).reset_index()

# NA, not 0.0, where nothing was determined — mirroring eligible_count(), which
# returns None rather than a rate over an empty set. A 0.0 there would assert
# "none of the determined rows are eligible" about no determined rows at all.
type_summary["pct_eligible_of_determined"] = (
    type_summary["nmtc_eligible"] / type_summary["determined"] * 100
).round(1)

print("Portfolio by Project Type:")
print(type_summary.to_string(index=False))

Portfolio by Project Type:
 project_type  count  total_amount  nmtc_eligible  determined  pct_eligible_of_determined
Commercial RE      1       8000000              0           1                         0.0
 Data Quality      1        750000              0           0                         NaN
  Food Access      1       1200000              0           1                         0.0
   Healthcare      1       3500000              1           1                       100.0
Manufacturing      1       4500000              1           1                       100.0
    Mixed-Use      1       6000000              1           1                       100.0


## 8. Export Results

In [9]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = os.path.join(tmpdir, "nmtc_eligibility_results.csv")
    enriched.to_csv(csv_path, index=False)
    print(f"Exported {len(enriched)} rows to CSV")

    reloaded = pd.read_csv(csv_path)
    print(f"Columns exported: {list(reloaded.columns)}")
    print(f"\nEligible projects in export:")
    eligible = reloaded[reloaded["nmtc_eligible"] == True]
    print(eligible[["project_name", "distress_level", "poverty_rate"]].to_string(index=False))


Exported 6 rows to CSV
Columns exported: ['project_name', 'project_type', 'amount', 'tract_id', 'nmtc_eligible', 'distress_level', 'poverty_rate', 'ami_ratio', 'unemployment_rate', 'is_non_metro', 'is_high_migration_rural', 'severe_distress', 'deep_distress', 'eligibility_status']

Eligible projects in export:
                     project_name distress_level  poverty_rate
Southside Community Health Center         severe         0.116
             Near North Mixed-Use            lic         0.276
     Autauga County Manufacturing            lic         0.170


## Summary

This notebook demonstrated the full nmtc-mapper workflow:

1. **Single tract lookup** — instant eligibility check by FIPS code
2. **Distress level analysis** — deep, severe, LIC, ineligible classification
3. **Batch enrichment** — enrich 10,000 rows in seconds using tract IDs
4. **Portfolio analysis** — eligibility and amount breakdowns by type
5. **Async geocoding** — convert addresses to tract IDs at scale
6. **Export** — CSV output for IC memos and grant applications

**Key advantage:** What previously required manual lookups in the CDFI Fund
CIMS tool one address at a time can now be done programmatically across
an entire portfolio in seconds.

**GitHub:** https://github.com/Jaypatel1511/nmtc-mapper
**Docs:** https://jaypatel1511.github.io/nmtc-mapper
**PyPI:** https://pypi.org/project/nmtc-mapper
